In [2]:
import torch
from torch_geometric.datasets import Planetoid

# Load the CORA dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')


/Users/rojankarki/Projects/LearningML/GNN/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = dataset[0]
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [4]:
print(f'Dataset: {dataset}')
print(f"Nodes (papers):        {data.num_nodes}")
print(f"Edges (citations):     {data.num_edges}")
print(f'Number of Classes (topics): {dataset.num_classes}')
print(f"Node feature dim:      {dataset.num_node_features}  (bag-of-words vocab)")
print(f"Training nodes:        {int(data.train_mask.sum())}  "
      f"({100*int(data.train_mask.sum())/data.num_nodes:.1f}% of all nodes labeled)")
print(f"Validation nodes:      {int(data.val_mask.sum())}")
print(f"Test nodes:            {int(data.test_mask.sum())}")


Dataset: Cora()
Nodes (papers):        2708
Edges (citations):     10556
Number of Classes (topics): 7
Node feature dim:      1433  (bag-of-words vocab)
Training nodes:        140  (5.2% of all nodes labeled)
Validation nodes:      500
Test nodes:            1000


In [5]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Layer 1: message pass + aggregate + update, then non-linearity
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # Layer 2: project down to class logits
        x = self.conv2(x, edge_index)
        return x


In [6]:
def train(model, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model, mask):
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)
    correct = (pred[mask] == data.y[mask]).sum()
    return int(correct) / int(mask.sum())

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

In [8]:
class TraditionalNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.layer1 = torch.nn.Linear(in_channels, hidden_channels)
        self.layer2 = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x = data.x
        x = self.layer1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.layer2(x)
        return x

# Training GCN with 16 channels

In [9]:
epochs = 200
model = GCN(
    in_channels=dataset.num_node_features,
    hidden_channels=16, 
    out_channels=dataset.num_classes
    ).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print("=== Training ===")
for epoch in range(epochs):
    loss = train(model, optimizer)
    val_acc = evaluate(model, data.val_mask)
    print(f"Epoch {epoch:3d} | Loss {loss:.4f} | Val Acc {val_acc:.4f}")


test_acc = evaluate(model, data.test_mask)
print(f"=== Final Test Accuracy: {test_acc:.4f} ===")

=== Training ===
Epoch   0 | Loss 1.9501 | Val Acc 0.4700
Epoch   1 | Loss 1.8491 | Val Acc 0.5940
Epoch   2 | Loss 1.7419 | Val Acc 0.6540
Epoch   3 | Loss 1.6090 | Val Acc 0.6940
Epoch   4 | Loss 1.5006 | Val Acc 0.7020
Epoch   5 | Loss 1.3763 | Val Acc 0.7280
Epoch   6 | Loss 1.2094 | Val Acc 0.7320
Epoch   7 | Loss 1.1234 | Val Acc 0.7440
Epoch   8 | Loss 1.0095 | Val Acc 0.7520
Epoch   9 | Loss 0.9155 | Val Acc 0.7620
Epoch  10 | Loss 0.8149 | Val Acc 0.7580
Epoch  11 | Loss 0.6813 | Val Acc 0.7620
Epoch  12 | Loss 0.6133 | Val Acc 0.7640
Epoch  13 | Loss 0.5683 | Val Acc 0.7660
Epoch  14 | Loss 0.4819 | Val Acc 0.7660
Epoch  15 | Loss 0.4281 | Val Acc 0.7700
Epoch  16 | Loss 0.3815 | Val Acc 0.7700
Epoch  17 | Loss 0.3366 | Val Acc 0.7760
Epoch  18 | Loss 0.3178 | Val Acc 0.7740
Epoch  19 | Loss 0.2680 | Val Acc 0.7780
Epoch  20 | Loss 0.3093 | Val Acc 0.7780
Epoch  21 | Loss 0.2433 | Val Acc 0.7760
Epoch  22 | Loss 0.1999 | Val Acc 0.7740
Epoch  23 | Loss 0.1806 | Val Acc 0.7740

# Training Linear NN with 16 channels

In [10]:
epochs = 200
model = TraditionalNN(
    in_channels=dataset.num_node_features,
    hidden_channels=16, 
    out_channels=dataset.num_classes
    ).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print("=== Training Traditional NN ===")
for epoch in range(epochs):
    loss = train(model, optimizer)
    val_acc = evaluate(model, data.val_mask)
    print(f"Epoch {epoch:3d} | Loss {loss:.4f} | Val Acc {val_acc:.4f}")


test_acc = evaluate(model, data.test_mask)
print(f"=== Final Test Accuracy: {test_acc:.4f} ===")

=== Training Traditional NN ===
Epoch   0 | Loss 1.9585 | Val Acc 0.1280
Epoch   1 | Loss 1.8929 | Val Acc 0.2340
Epoch   2 | Loss 1.8197 | Val Acc 0.3160
Epoch   3 | Loss 1.7258 | Val Acc 0.3720
Epoch   4 | Loss 1.6078 | Val Acc 0.4180
Epoch   5 | Loss 1.5239 | Val Acc 0.4300
Epoch   6 | Loss 1.4180 | Val Acc 0.4560
Epoch   7 | Loss 1.2805 | Val Acc 0.4780
Epoch   8 | Loss 1.2006 | Val Acc 0.4920
Epoch   9 | Loss 1.0527 | Val Acc 0.4940
Epoch  10 | Loss 1.0315 | Val Acc 0.5020
Epoch  11 | Loss 1.0040 | Val Acc 0.5080
Epoch  12 | Loss 0.8717 | Val Acc 0.5060
Epoch  13 | Loss 0.8705 | Val Acc 0.5100
Epoch  14 | Loss 0.6628 | Val Acc 0.5100
Epoch  15 | Loss 0.7354 | Val Acc 0.5180
Epoch  16 | Loss 0.6053 | Val Acc 0.5180
Epoch  17 | Loss 0.6090 | Val Acc 0.5180
Epoch  18 | Loss 0.5368 | Val Acc 0.5200
Epoch  19 | Loss 0.4923 | Val Acc 0.5240
Epoch  20 | Loss 0.4554 | Val Acc 0.5200
Epoch  21 | Loss 0.4631 | Val Acc 0.5200
Epoch  22 | Loss 0.4288 | Val Acc 0.5220
Epoch  23 | Loss 0.4207 |

# Comparing results with different channel configs

In [11]:
epochs = 200
result = []
hidden_channels = [8, 16, 32, 64, 128]
print("=== Training ===")
for hidden in hidden_channels:
    model = GCN(
        in_channels=dataset.num_node_features,
        hidden_channels=hidden, 
        out_channels=dataset.num_classes
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for epoch in range(epochs):
        loss = train(model, optimizer)
        if (epoch+1) % 100 == 0:
                val_acc = evaluate(model, data.val_mask)
                print(f"Epoch {epoch:3d} | Loss {loss:.4f} | Val Acc {val_acc:.4f}")    
    test_acc = evaluate(model, data.test_mask)
    print(f"=== Channel={hidden}, Final Test Accuracy: {test_acc:.4f} ===")
    output = {'hidden_channels': hidden, 'validation_accuracy':val_acc, 'test_accuracy': test_acc}
    result.append(output)


=== Training ===
Epoch  99 | Loss 0.1178 | Val Acc 0.7400
Epoch 199 | Loss 0.1057 | Val Acc 0.7480
=== Channel=8, Final Test Accuracy: 0.7760 ===
Epoch  99 | Loss 0.0342 | Val Acc 0.7720
Epoch 199 | Loss 0.0217 | Val Acc 0.7680
=== Channel=16, Final Test Accuracy: 0.7950 ===
Epoch  99 | Loss 0.0183 | Val Acc 0.7740
Epoch 199 | Loss 0.0120 | Val Acc 0.7700
=== Channel=32, Final Test Accuracy: 0.8010 ===
Epoch  99 | Loss 0.0122 | Val Acc 0.7720
Epoch 199 | Loss 0.0091 | Val Acc 0.7700
=== Channel=64, Final Test Accuracy: 0.8080 ===
Epoch  99 | Loss 0.0085 | Val Acc 0.7680
Epoch 199 | Loss 0.0081 | Val Acc 0.7720
=== Channel=128, Final Test Accuracy: 0.8040 ===


In [12]:
result

[{'hidden_channels': 8, 'validation_accuracy': 0.748, 'test_accuracy': 0.776},
 {'hidden_channels': 16, 'validation_accuracy': 0.768, 'test_accuracy': 0.795},
 {'hidden_channels': 32, 'validation_accuracy': 0.77, 'test_accuracy': 0.801},
 {'hidden_channels': 64, 'validation_accuracy': 0.77, 'test_accuracy': 0.808},
 {'hidden_channels': 128,
  'validation_accuracy': 0.772,
  'test_accuracy': 0.804}]